# Pretraining -- Build: Pretrain a Small Language Model

> **Section 01** -- Topic 05 -- `build`  
> **Prerequisites:** `foundations.ipynb`, `advanced.ipynb`, `expert.ipynb`  
> **Objective:** Pretrain a ~10M parameter language model from scratch on a real corpus, implementing the full pipeline: model definition, data preparation, training loop with all modern best practices, evaluation, and text generation.

---

This notebook is the culmination of everything we have learned about pretraining. We go from raw text to a working language model in a single session. Every component is written from scratch -- no training frameworks, no abstractions hiding the details.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np
import time
import os
from dataclasses import dataclass
from typing import Optional

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
dtype = torch.bfloat16 if (device.type == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float32
print(f"Device: {device}")
print(f"Dtype: {dtype}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 1. Define Model Architecture

We build a **Llama-style** architecture with all modern components:

| Component | Choice | Why |
|-----------|--------|-----|
| Position encoding | RoPE | No learned parameters, length generalizable |
| Normalization | RMSNorm (pre-norm) | Faster than LayerNorm, stable training |
| Activation | SwiGLU | Better loss per parameter than GELU |
| Attention | Grouped-Query (GQA) | Smaller KV cache at inference |
| Bias | None | Slight regularization, fewer params |
| Weight tying | Embedding = LM head | Saves ~20% params at this scale |

Target: **~10M parameters** -- large enough to learn real language patterns, small enough to train in minutes on a single GPU.

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int = 50257       # GPT-2 tokenizer
    max_seq_len: int = 256        # Context window
    d_model: int = 384            # Hidden dimension
    n_layers: int = 6             # Transformer blocks
    n_heads: int = 6              # Query heads
    n_kv_heads: int = 2           # KV heads (GQA ratio = 3:1)
    dropout: float = 0.0          # No dropout (modern practice)
    
    @property
    def d_head(self):
        return self.d_model // self.n_heads
    
    @property
    def ffn_hidden(self):
        """SwiGLU hidden dim: 8/3 * d_model, rounded to nearest 64."""
        h = int(8 / 3 * self.d_model)
        return 64 * ((h + 63) // 64)


config = ModelConfig()
print(f"Config: {config}")
print(f"Head dim: {config.d_head}")
print(f"FFN hidden dim: {config.ffn_hidden}")
print(f"GQA ratio: {config.n_heads // config.n_kv_heads}:1")

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).type_as(x) * self.weight

In [ ]:
def precompute_rope(d_head: int, max_len: int = 4096, theta: float = 10000.0):
    """Precompute RoPE cos/sin tables."""
    freqs = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    positions = torch.arange(max_len).float()
    angles = torch.outer(positions, freqs)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x, cos, sin):
    """Apply RoPE to (B, H, T, d_k) tensor."""
    B, H, T, d_k = x.shape
    pairs = x.float().reshape(B, H, T, d_k // 2, 2)
    r, i = pairs[..., 0], pairs[..., 1]
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    out = torch.stack([r * c - i * s, r * s + i * c], dim=-1)
    return out.reshape(B, H, T, d_k).type_as(x)

In [ ]:
class GQAttention(nn.Module):
    """Grouped-Query Attention with RoPE."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.d_head = cfg.d_head
        self.n_rep = cfg.n_heads // cfg.n_kv_heads
        
        self.wq = nn.Linear(cfg.d_model, cfg.n_heads * cfg.d_head, bias=False)
        self.wk = nn.Linear(cfg.d_model, cfg.n_kv_heads * cfg.d_head, bias=False)
        self.wv = nn.Linear(cfg.d_model, cfg.n_kv_heads * cfg.d_head, bias=False)
        self.wo = nn.Linear(cfg.n_heads * cfg.d_head, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, mask):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        
        # Apply RoPE to Q and K
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        
        # Repeat KV heads for GQA
        if self.n_rep > 1:
            k = k[:, :, None, :, :].expand(B, self.n_kv_heads, self.n_rep, T, self.d_head).reshape(B, self.n_heads, T, self.d_head)
            v = v[:, :, None, :, :].expand(B, self.n_kv_heads, self.n_rep, T, self.d_head).reshape(B, self.n_heads, T, self.d_head)
        
        # Scaled dot-product attention
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        scores = scores.masked_fill(mask[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, -1)
        return self.wo(out)


class SwiGLU(nn.Module):
    """SwiGLU feed-forward network."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        h = cfg.ffn_hidden
        self.w_gate = nn.Linear(cfg.d_model, h, bias=False)
        self.w_up = nn.Linear(cfg.d_model, h, bias=False)
        self.w_down = nn.Linear(h, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class TransformerBlock(nn.Module):
    """Pre-RMSNorm transformer block."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)

    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.norm1(x), cos, sin, mask)
        x = x + self.ffn(self.norm2(x))
        return x

In [ ]:
class SmallLM(nn.Module):
    """Complete language model for pretraining."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        
        # Weight tying -- embedding and LM head share parameters
        self.tok_emb.weight = self.lm_head.weight
        
        # Precompute RoPE
        cos, sin = precompute_rope(cfg.d_head, cfg.max_seq_len)
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)
        
        # Causal mask
        self.register_buffer('causal_mask', torch.tril(torch.ones(1, 1, cfg.max_seq_len, cfg.max_seq_len)))
        
        # Initialize weights
        self.apply(self._init_weights)
        # Scale residual projections by 1/sqrt(2*n_layers) for training stability
        for block in self.blocks:
            nn.init.normal_(block.attn.wo.weight, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))
            nn.init.normal_(block.ffn.w_down.weight, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)
        
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin, self.causal_mask)
        
        x = self.norm_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


model = SmallLM(config)
print(f"Model created successfully.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 2. Parameter Breakdown

Understanding where parameters live helps with scaling decisions. In a transformer, the majority of parameters are in the FFN layers (roughly 2/3) and attention projections (roughly 1/3), with the embedding table being shared with the output head.

In [ ]:
def detailed_param_breakdown(model):
    """Detailed parameter count by module type."""
    categories = {
        'Embeddings': 0,
        'Attention (Q)': 0,
        'Attention (KV)': 0,
        'Attention (O)': 0,
        'FFN (SwiGLU)': 0,
        'RMSNorm': 0,
    }
    
    for name, param in model.named_parameters():
        n = param.numel()
        if 'tok_emb' in name:
            categories['Embeddings'] += n
        elif 'wq' in name:
            categories['Attention (Q)'] += n
        elif 'wk' in name or 'wv' in name:
            categories['Attention (KV)'] += n
        elif 'wo' in name:
            categories['Attention (O)'] += n
        elif 'w_gate' in name or 'w_up' in name or 'w_down' in name:
            categories['FFN (SwiGLU)'] += n
        elif 'norm' in name:
            categories['RMSNorm'] += n
    
    total = sum(categories.values())
    print(f"{'Component':<25s} {'Params':>12s} {'Pct':>8s}")
    print("=" * 48)
    for cat, count in categories.items():
        pct = 100 * count / total
        print(f"{cat:<25s} {count:>12,} {pct:>7.1f}%")
    print("=" * 48)
    print(f"{'TOTAL':<25s} {total:>12,}")
    print(f"\nNote: Embeddings and LM head are weight-tied (counted once).")
    return categories


cats = detailed_param_breakdown(model)

In [ ]:
# Visualize parameter distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
labels = list(cats.keys())
sizes = list(cats.values())
colors = ['#3498db', '#e74c3c', '#f39c12', '#2ecc71', '#9b59b6', '#1abc9c']
wedges, texts, autotexts = ax1.pie(sizes, labels=labels, autopct='%1.1f%%',
                                    colors=colors, textprops={'fontsize': 9})
ax1.set_title('Parameter Distribution', fontsize=13, fontweight='bold')

# Bar chart -- per-layer breakdown
layer_params = []
for i in range(config.n_layers):
    block_params = sum(p.numel() for n, p in model.named_parameters() if f'blocks.{i}.' in n)
    layer_params.append(block_params)

ax2.bar(range(config.n_layers), layer_params, color='#3498db', edgecolor='black', linewidth=0.5)
ax2.set_xlabel('Layer Index')
ax2.set_ylabel('Parameters')
ax2.set_title('Parameters per Transformer Block', fontsize=13, fontweight='bold')
ax2.set_xticks(range(config.n_layers))
for i, v in enumerate(layer_params):
    ax2.text(i, v + 1000, f'{v:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 3. Prepare Dataset

We use **TinyStories** (Eldan & Li, 2023) -- a dataset of short children's stories generated by GPT-4. It was specifically designed to train small language models that produce coherent, grammatical text.

Why TinyStories is perfect for us:
- **Small vocabulary** -- simple words that a small model can learn
- **Short stories** -- complete narratives that fit in our 256-token context
- **Diverse patterns** -- varied sentence structures despite simple language
- **High quality** -- generated by GPT-4, so grammar and coherence are excellent

If TinyStories is unavailable, we fall back to WikiText-2.

In [ ]:
from transformers import GPT2Tokenizer
from datasets import load_dataset

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Try TinyStories first, fall back to WikiText-2
try:
    dataset = load_dataset('roneneldan/TinyStories', split='train[:50000]')
    val_dataset = load_dataset('roneneldan/TinyStories', split='validation[:5000]')
    text_key = 'text'
    dataset_name = 'TinyStories'
    print(f"Loaded TinyStories: {len(dataset)} train, {len(val_dataset)} val")
except Exception:
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1')
    dataset = ds['train']
    val_dataset = ds['validation']
    text_key = 'text'
    dataset_name = 'WikiText-2'
    print(f"Loaded WikiText-2: {len(dataset)} train, {len(val_dataset)} val")

print(f"\nDataset: {dataset_name}")
print(f"Sample: {dataset[0][text_key][:300]}")

In [ ]:
def tokenize_and_pack(split, text_key, seq_len=256, max_tokens=None):
    """Tokenize all texts and pack into fixed-length training sequences.
    
    Packing: concatenate all tokens into one stream, then chunk into
    non-overlapping sequences of length seq_len. This wastes no tokens
    on padding and is standard practice for pretraining.
    """
    all_tokens = []
    eos = tokenizer.eos_token_id
    
    for example in split:
        text = example[text_key]
        if text.strip():
            tokens = tokenizer.encode(text)
            all_tokens.extend(tokens)
            all_tokens.append(eos)  # Separate documents with EOS
            
            if max_tokens and len(all_tokens) >= max_tokens:
                all_tokens = all_tokens[:max_tokens]
                break
    
    all_tokens = torch.tensor(all_tokens, dtype=torch.long)
    n_chunks = (len(all_tokens) - 1) // seq_len
    all_tokens = all_tokens[:n_chunks * seq_len + 1]
    
    inputs = all_tokens[:-1].reshape(n_chunks, seq_len)
    targets = all_tokens[1:].reshape(n_chunks, seq_len)
    
    print(f"  Total tokens: {len(all_tokens):,}")
    print(f"  Sequences: {n_chunks:,} x {seq_len}")
    return inputs, targets


print("Tokenizing training data...")
train_x, train_y = tokenize_and_pack(dataset, text_key, seq_len=config.max_seq_len)

print("\nTokenizing validation data...")
val_x, val_y = tokenize_and_pack(val_dataset, text_key, seq_len=config.max_seq_len)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

batch_size = 64

train_loader = DataLoader(
    TensorDataset(train_x, train_y),
    batch_size=batch_size, shuffle=True, drop_last=True,
    num_workers=0, pin_memory=(device.type == 'cuda')
)
val_loader = DataLoader(
    TensorDataset(val_x, val_y),
    batch_size=batch_size, shuffle=False, drop_last=True
)

print(f"Train batches: {len(train_loader)} (batch_size={batch_size})")
print(f"Val batches:   {len(val_loader)}")
print(f"Tokens per batch: {batch_size * config.max_seq_len:,}")

# Peek at data
xb, yb = next(iter(train_loader))
print(f"\nBatch shape: {xb.shape}")
print(f"Sample text: {tokenizer.decode(xb[0][:50].tolist())}...")

---
## 4. Configure Training

Modern LLM pretraining uses a specific set of well-tested hyperparameters. Here we follow the recipe from Llama/Chinchilla:

- **AdamW** with $\beta_1 = 0.9$, $\beta_2 = 0.95$, $\epsilon = 10^{-8}$
- **Weight decay** = 0.1 (only on 2D parameters, not norms/biases)
- **Cosine LR schedule** from `max_lr` down to `max_lr / 10`
- **Linear warmup** for 10% of training steps
- **Gradient clipping** at norm 1.0
- **Mixed precision** (bf16) when available

### Why these specific values?

- **$\beta_2 = 0.95$** (not 0.999): LLM training has non-stationary gradients; lower $\beta_2$ adapts faster
- **Weight decay = 0.1**: strong regularization needed because we see each token only ~3 times
- **Cosine schedule**: smoothly reduces LR, empirically better than step decay
- **Grad clip = 1.0**: prevents exploding gradients during early training instability

In [ ]:
# Training hyperparameters
max_lr = 3e-4
min_lr = 3e-5      # max_lr / 10
n_epochs = 3
warmup_frac = 0.1
weight_decay = 0.1
grad_clip = 1.0
log_interval = 25   # Log every N steps
eval_interval = 100  # Evaluate every N steps
checkpoint_interval = 500  # Save checkpoint every N steps

max_steps = n_epochs * len(train_loader)
warmup_steps = int(warmup_frac * max_steps)

print(f"Training configuration:")
print(f"  Max steps: {max_steps}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Tokens per step: {batch_size * config.max_seq_len:,}")
print(f"  Total tokens: {max_steps * batch_size * config.max_seq_len:,}")
print(f"  LR: {max_lr} -> {min_lr} (cosine)")
print(f"  Weight decay: {weight_decay}")
print(f"  Gradient clipping: {grad_clip}")

In [ ]:
def get_cosine_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    """Cosine learning rate with linear warmup."""
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


# Visualize schedule
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

steps = range(max_steps)
lrs = [get_cosine_lr(s, warmup_steps, max_steps, max_lr, min_lr) for s in steps]

ax1.plot(steps, lrs, linewidth=2, color='#e74c3c')
ax1.axvline(warmup_steps, color='gray', linestyle='--', alpha=0.5, label=f'Warmup ({warmup_steps} steps)')
ax1.set_xlabel('Step')
ax1.set_ylabel('Learning Rate')
ax1.set_title('Learning Rate Schedule')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log scale
ax2.semilogy(steps, lrs, linewidth=2, color='#3498db')
ax2.axvline(warmup_steps, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Step')
ax2.set_ylabel('Learning Rate (log scale)')
ax2.set_title('LR Schedule (Log Scale)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Set up optimizer with separate weight decay groups
model = model.to(device)

# Only apply weight decay to 2D parameters (weights, not norms/embeddings)
decay_params = [p for n, p in model.named_parameters() if p.dim() >= 2 and p.requires_grad]
no_decay_params = [p for n, p in model.named_parameters() if p.dim() < 2 and p.requires_grad]

optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': weight_decay},
    {'params': no_decay_params, 'weight_decay': 0.0}
], lr=max_lr, betas=(0.9, 0.95), eps=1e-8)

print(f"Optimizer groups:")
print(f"  Decay params:    {sum(p.numel() for p in decay_params):>10,} (wd={weight_decay})")
print(f"  No-decay params: {sum(p.numel() for p in no_decay_params):>10,} (wd=0.0)")

---
## 5. Training Loop

The training loop is the heart of pretraining. We track:
- **Loss** per step and per epoch
- **Learning rate** at each step
- **Throughput** in tokens per second
- **Gradient norm** to detect training instabilities
- **Validation loss and perplexity** periodically

In [ ]:
@torch.no_grad()
def evaluate_model(model, val_loader, device, dtype):
    """Compute validation loss and perplexity."""
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for xb, yb in val_loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=(dtype != torch.float32)):
            _, loss = model(xb, yb)
        total_loss += loss.item()
        n_batches += 1
    avg_loss = total_loss / max(n_batches, 1)
    return avg_loss, math.exp(avg_loss)

In [ ]:
# Pre-training evaluation (sanity check)
init_loss, init_ppl = evaluate_model(model, val_loader, device, dtype)
expected_init_loss = math.log(config.vocab_size)  # ~10.8 for uniform distribution
print(f"Initial validation loss: {init_loss:.4f} (expected ~{expected_init_loss:.1f} for random model)")
print(f"Initial perplexity: {init_ppl:.0f} (expected ~{config.vocab_size} for random model)")
print(f"\nThe initial loss should be close to ln(vocab_size) = {expected_init_loss:.4f}.")
print(f"If it is much higher or lower, there may be a bug in the model or data pipeline.")

In [ ]:
# ================================================================
# MAIN TRAINING LOOP
# ================================================================

use_amp = dtype != torch.float32
scaler = torch.amp.GradScaler(enabled=(use_amp and device.type == 'cuda'))

# Logging
history = {
    'step': [], 'train_loss': [], 'val_loss': [], 'val_ppl': [],
    'lr': [], 'grad_norm': [], 'tokens_per_sec': []
}

step = 0
best_val_loss = float('inf')
t_start = time.time()

print(f"Starting pretraining...")
print(f"{'='*80}")

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0.0
    epoch_steps = 0
    t_epoch = time.time()
    
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        # Update learning rate
        lr = get_cosine_lr(step, warmup_steps, max_steps, max_lr, min_lr)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
        
        # Forward
        t_step = time.time()
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
            _, loss = model(xb, yb)
        
        # Backward
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        epoch_steps += 1
        dt = time.time() - t_step
        tokens_per_sec = xb.shape[0] * xb.shape[1] / max(dt, 1e-6)
        
        # Log
        if step % log_interval == 0:
            history['step'].append(step)
            history['train_loss'].append(loss.item())
            history['lr'].append(lr)
            history['grad_norm'].append(grad_norm.item() if isinstance(grad_norm, torch.Tensor) else grad_norm)
            history['tokens_per_sec'].append(tokens_per_sec)
        
        # Evaluate
        if step % eval_interval == 0:
            val_loss, val_ppl = evaluate_model(model, val_loader, device, dtype)
            history['val_loss'].append(val_loss)
            history['val_ppl'].append(val_ppl)
            
            is_best = val_loss < best_val_loss
            if is_best:
                best_val_loss = val_loss
            
            elapsed = time.time() - t_start
            print(f"  Step {step:5d}/{max_steps} | Loss: {loss.item():.4f} | Val: {val_loss:.4f} | "
                  f"PPL: {val_ppl:7.1f} | LR: {lr:.2e} | GradNorm: {grad_norm:.2f} | "
                  f"{tokens_per_sec:,.0f} tok/s | {elapsed:.0f}s"
                  f"{' *' if is_best else ''}")
            model.train()
        
        step += 1
    
    # End-of-epoch summary
    avg_epoch_loss = epoch_loss / epoch_steps
    val_loss, val_ppl = evaluate_model(model, val_loader, device, dtype)
    dt_epoch = time.time() - t_epoch
    print(f"\n  Epoch {epoch+1}/{n_epochs} complete in {dt_epoch:.0f}s")
    print(f"  Avg train loss: {avg_epoch_loss:.4f} | Val loss: {val_loss:.4f} | Val PPL: {val_ppl:.1f}")
    print(f"{'='*80}")

total_time = time.time() - t_start
total_tokens = step * batch_size * config.max_seq_len
print(f"\nTraining complete!")
print(f"  Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"  Total tokens processed: {total_tokens:,}")
print(f"  Average throughput: {total_tokens/total_time:,.0f} tokens/sec")
print(f"  Best val loss: {best_val_loss:.4f} (PPL: {math.exp(best_val_loss):.1f})")

---
## 6. Training Diagnostics

Visualizing training metrics is essential for diagnosing issues. We look for:
- **Smooth loss curve**: spikes indicate data issues or LR problems
- **Decreasing validation loss**: ensures we are not overfitting
- **Stable gradient norms**: exploding gradients = training instability
- **Reasonable throughput**: should be stable after warmup

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Training loss
ax = axes[0, 0]
ax.plot(history['step'], history['train_loss'], alpha=0.6, color='#3498db', linewidth=0.8)
# Smoothed
window = min(20, len(history['train_loss']) // 4)
if window > 1:
    smoothed = np.convolve(history['train_loss'], np.ones(window)/window, mode='valid')
    ax.plot(history['step'][window-1:], smoothed, color='#e74c3c', linewidth=2, label='Smoothed')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Validation loss
ax = axes[0, 1]
val_steps = [history['step'][i] for i in range(0, len(history['step']), eval_interval // log_interval)][:len(history['val_loss'])]
if history['val_loss']:
    ax.plot(val_steps, history['val_loss'], 'o-', color='#e74c3c', markersize=4)
ax.set_xlabel('Step')
ax.set_ylabel('Validation Loss')
ax.set_title('Validation Loss')
ax.grid(True, alpha=0.3)

# 3. Validation perplexity
ax = axes[0, 2]
if history['val_ppl']:
    ax.plot(val_steps, history['val_ppl'], 's-', color='#2ecc71', markersize=4)
ax.set_xlabel('Step')
ax.set_ylabel('Perplexity')
ax.set_title('Validation Perplexity')
ax.grid(True, alpha=0.3)

# 4. Learning rate
ax = axes[1, 0]
ax.plot(history['step'], history['lr'], color='#f39c12', linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.grid(True, alpha=0.3)

# 5. Gradient norm
ax = axes[1, 1]
ax.plot(history['step'], history['grad_norm'], alpha=0.6, color='#9b59b6', linewidth=0.8)
ax.axhline(y=grad_clip, color='red', linestyle='--', alpha=0.5, label=f'Clip threshold ({grad_clip})')
ax.set_xlabel('Step')
ax.set_ylabel('Gradient Norm')
ax.set_title('Gradient Norm')
ax.legend()
ax.grid(True, alpha=0.3)

# 6. Throughput
ax = axes[1, 2]
ax.plot(history['step'], [t/1000 for t in history['tokens_per_sec']], alpha=0.6, color='#1abc9c', linewidth=0.8)
ax.set_xlabel('Step')
ax.set_ylabel('Throughput (K tokens/sec)')
ax.set_title('Training Throughput')
ax.grid(True, alpha=0.3)

plt.suptitle('Pretraining Diagnostics', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Evaluate on Validation Set

Final evaluation with detailed metrics.

In [ ]:
final_val_loss, final_val_ppl = evaluate_model(model, val_loader, device, dtype)

print(f"{'='*50}")
print(f"Final Evaluation Results")
print(f"{'='*50}")
print(f"  Validation loss:      {final_val_loss:.4f}")
print(f"  Validation perplexity: {final_val_ppl:.1f}")
print(f"  Initial loss was:      {init_loss:.4f}")
print(f"  Loss reduction:        {init_loss - final_val_loss:.4f} ({(init_loss - final_val_loss)/init_loss*100:.1f}%)")
print(f"  PPL reduction:         {init_ppl:.0f} -> {final_val_ppl:.1f}")
print(f"{'='*50}")

---
## 8. Text Generation

The true test of a language model: can it generate coherent text? We implement top-k / top-p sampling and generate at multiple temperatures to see how the model behaves.

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=150, temperature=0.8,
             top_k=50, top_p=0.9, device='cpu'):
    """Generate text with top-k and top-p (nucleus) sampling."""
    model.eval()
    tokens = tokenizer.encode(prompt)
    tokens = torch.tensor([tokens], dtype=torch.long, device=device)
    
    for _ in range(max_new_tokens):
        idx = tokens[:, -model.cfg.max_seq_len:]
        logits, _ = model(idx)
        logits = logits[:, -1, :] / max(temperature, 1e-8)
        
        # Top-k
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
        
        # Top-p (nucleus)
        if top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cumprobs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cumprobs - F.softmax(sorted_logits, dim=-1) > top_p
            sorted_logits[mask] = float('-inf')
            logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)
        
        probs = F.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1)
        tokens = torch.cat([tokens, next_tok], dim=1)
        
        if next_tok.item() == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(tokens[0].tolist())

In [ ]:
# Generate at different temperatures
prompts = ["Once upon a time", "The little girl", "One day, a"]
temperatures = [0.5, 0.8, 1.2]

for prompt in prompts:
    print(f"\n{'='*70}")
    print(f"PROMPT: \"{prompt}\"")
    print(f"{'='*70}")
    
    for temp in temperatures:
        text = generate(model, tokenizer, prompt, max_new_tokens=80,
                       temperature=temp, top_k=50, top_p=0.9, device=device)
        print(f"\n  Temperature {temp}:")
        print(f"  {text}")
        print()

---
## 9. Scaling Law Comparison

Kaplan et al. (2020) and Hoffmann et al. (2022, "Chinchilla") established **scaling laws** that predict language model performance as a function of model size and training data. Let us see how our model compares.

The Chinchilla scaling law suggests:

$$L(N, D) = \frac{A}{N^\alpha} + \frac{B}{D^\beta} + E$$

where $N$ is model params, $D$ is training tokens, and $E \approx 1.69$ is the irreducible loss.

In [ ]:
# Chinchilla scaling law prediction
# Parameters from Hoffmann et al. 2022
A = 406.4
B = 410.7
alpha = 0.34
beta = 0.28
E = 1.69  # Irreducible loss (natural language entropy)

N = sum(p.numel() for p in model.parameters())  # Model params
D = step * batch_size * config.max_seq_len  # Training tokens

predicted_loss = A / (N ** alpha) + B / (D ** beta) + E

print(f"Scaling Law Comparison")
print(f"{'='*50}")
print(f"  Model size (N): {N:,} parameters")
print(f"  Training tokens (D): {D:,}")
print(f"  Chinchilla-optimal D for this N: {20 * N:,}")
print(f"  Our D/N ratio: {D/N:.1f} (Chinchilla suggests ~20)")
print(f"")
print(f"  Predicted loss (Chinchilla): {predicted_loss:.4f}")
print(f"  Actual validation loss:      {final_val_loss:.4f}")
print(f"  Difference:                  {final_val_loss - predicted_loss:+.4f}")
print(f"")
print(f"  Note: Scaling laws were fit on much larger models (125M-500B params)")
print(f"  and may not extrapolate perfectly to our ~10M model.")

In [ ]:
# Visualize where our model sits on the scaling curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss vs model size (holding D constant)
model_sizes = np.logspace(6, 11, 50)  # 1M to 100B
losses_by_size = [A / (n ** alpha) + B / (D ** beta) + E for n in model_sizes]

ax1.semilogx(model_sizes, losses_by_size, 'b-', linewidth=2, label='Chinchilla prediction')
ax1.semilogx(N, final_val_loss, 'r*', markersize=15, label=f'Our model ({N/1e6:.1f}M params)')
ax1.set_xlabel('Model Parameters')
ax1.set_ylabel('Validation Loss')
ax1.set_title(f'Loss vs Model Size (D={D:,} tokens)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axhline(y=E, color='gray', linestyle=':', alpha=0.5, label=f'Irreducible loss ({E})')

# Loss vs training tokens (holding N constant)
token_counts = np.logspace(5, 11, 50)  # 100K to 100B
losses_by_tokens = [A / (N ** alpha) + B / (d ** beta) + E for d in token_counts]

ax2.semilogx(token_counts, losses_by_tokens, 'b-', linewidth=2, label='Chinchilla prediction')
ax2.semilogx(D, final_val_loss, 'r*', markersize=15, label=f'Our model ({D/1e6:.1f}M tokens)')
ax2.set_xlabel('Training Tokens')
ax2.set_ylabel('Validation Loss')
ax2.set_title(f'Loss vs Training Tokens (N={N:,})')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(y=E, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

---
## 10. Token-Level Analysis

Let us examine what the model has learned by looking at per-token predictions. This reveals which types of tokens the model predicts well (function words, common patterns) vs. poorly (rare words, proper nouns).

In [ ]:
@torch.no_grad()
def analyze_predictions(model, text, tokenizer, device):
    """Analyze per-token predictions and losses."""
    model.eval()
    tokens = tokenizer.encode(text)
    input_ids = torch.tensor([tokens[:-1]], device=device)
    target_ids = torch.tensor([tokens[1:]], device=device)
    
    logits, _ = model(input_ids)
    probs = F.softmax(logits, dim=-1)
    
    results = []
    for i in range(len(tokens) - 1):
        target_token = tokens[i + 1]
        target_prob = probs[0, i, target_token].item()
        target_loss = -math.log(max(target_prob, 1e-10))
        top5_probs, top5_ids = probs[0, i].topk(5)
        top5_tokens = [tokenizer.decode([t.item()]) for t in top5_ids]
        
        results.append({
            'context': tokenizer.decode([tokens[i]]),
            'target': tokenizer.decode([target_token]),
            'prob': target_prob,
            'loss': target_loss,
            'top5': list(zip(top5_tokens, top5_probs.tolist()))
        })
    
    return results


# Analyze a sample
sample_text = "Once upon a time there was a little girl who loved to play in the garden"
results = analyze_predictions(model, sample_text, tokenizer, device)

print(f"Per-token prediction analysis:")
print(f"{'Context':<12s} {'Target':<12s} {'Prob':>8s} {'Loss':>8s}  Top predictions")
print("-" * 80)
for r in results:
    top_str = ', '.join(f"{t}({p:.2f})" for t, p in r['top5'][:3])
    print(f"{r['context']:<12s} {r['target']:<12s} {r['prob']:>8.3f} {r['loss']:>8.3f}  {top_str}")

In [ ]:
# Visualize per-token losses
token_labels = [r['target'] for r in results]
losses = [r['loss'] for r in results]
probs = [r['prob'] for r in results]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8))

# Loss per token
colors = ['#2ecc71' if l < 2 else '#f39c12' if l < 5 else '#e74c3c' for l in losses]
ax1.bar(range(len(losses)), losses, color=colors, edgecolor='black', linewidth=0.3)
ax1.set_xticks(range(len(token_labels)))
ax1.set_xticklabels(token_labels, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('Loss (nats)')
ax1.set_title('Per-Token Loss (green=easy, red=hard)')
ax1.grid(True, alpha=0.3, axis='y')

# Probability per token
ax2.bar(range(len(probs)), probs, color='#3498db', edgecolor='black', linewidth=0.3)
ax2.set_xticks(range(len(token_labels)))
ax2.set_xticklabels(token_labels, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Probability')
ax2.set_title('Per-Token Predicted Probability')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## Summary

In this notebook we pretrained a language model completely from scratch:

1. **Defined a modern ~10M parameter architecture** with RoPE, RMSNorm, SwiGLU, and GQA -- the same recipe used by Llama 3 at massive scale
2. **Prepared a real training dataset** with proper tokenization and sequence packing
3. **Implemented a production-quality training loop** with AdamW, cosine LR schedule, gradient clipping, mixed precision, and comprehensive logging
4. **Trained the model** and tracked loss, perplexity, gradient norms, and throughput
5. **Evaluated rigorously** -- validation perplexity, text generation at multiple temperatures, scaling law comparison, and per-token analysis

### Key insights

- Even at 10M parameters, the model learns meaningful language patterns
- The cosine LR schedule with warmup provides stable training
- Gradient clipping is essential -- without it, training often diverges
- Temperature controls the diversity-quality tradeoff in generation
- Our model roughly follows Chinchilla scaling predictions, even at this small scale

---

**Next:** `../06-post-training/foundations.ipynb` -- How to transform a pretrained base model into a helpful assistant through supervised fine-tuning and alignment.